In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
GROQ_API_KEY=os.getenv("GROQ_API_KEY")

from langchain_groq import ChatGroq

model=ChatGroq(model="llama-3.3-70b-versatile",groq_api_key=GROQ_API_KEY)

In [17]:
from langchain_core.messages import SystemMessage,trim_messages,HumanMessage,AIMessage

#   How many message we are sending to the model
trimmer=trim_messages(
    max_tokens=40,
    strategy="last",
    token_counter=model.get_num_tokens_from_messages,
    include_system=True,
    allow_partial=False,
    start_on="human"
    
)

In [14]:
messages=[
    SystemMessage(content="You are a good assistant"),
    HumanMessage(content="Hi, I am Bob"),
    AIMessage(content="Hi"),
    HumanMessage(content="Hi, I like Chocolate Ice-Cream !!"),
    AIMessage(content="Nice"),
    HumanMessage(content="What is 2+2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="No Problem"),
    HumanMessage(content="Having fun"),
    AIMessage(content="Yes!!")
    ]


trimmer.invoke(messages)



[SystemMessage(content='You are a good assistant', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hi, I am Bob', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hi', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Hi, I like Chocolate Ice-Cream !!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What is 2+2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='No Problem', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Having fun', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Yes!!', addition

In [15]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
       ("system","You are a helpful assistant,Answer all the question to the best of your ability in {language}"),
       MessagesPlaceholder(variable_name="messages") 
    ]
)

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
)


chain.invoke(
    {
        "messages" : messages + [HumanMessage(content="What ice-cream do i like?")],
        "language" : "English"
    }
)

AIMessage(content='You like Chocolate Ice-Cream!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 131, 'total_tokens': 139, 'completion_time': 0.02306423, 'completion_tokens_details': None, 'prompt_time': 0.012559851, 'prompt_tokens_details': None, 'queue_time': 0.165388647, 'total_time': 0.035624081}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fa84b-4170-7ed1-b95c-8adbf9211b8a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 131, 'output_tokens': 8, 'total_tokens': 139})

In [18]:
chain.invoke(
    {
        "messages" : messages + [HumanMessage(content="What mathematical problem did I ask?")],
        "language" : "English"
    }
)

AIMessage(content='You asked "What is 2+2" which is a basic addition problem. I answered that the result is 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 130, 'total_tokens': 157, 'completion_time': 0.086519319, 'completion_tokens_details': None, 'prompt_time': 0.00620171, 'prompt_tokens_details': None, 'queue_time': 0.052215879, 'total_time': 0.092721029}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fa84c-bdb5-78d3-9f93-02c7bb3388a0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 130, 'output_tokens': 27, 'total_tokens': 157})

In [19]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

sessionIdStore={}
# User Session
def getSessionHistory(session_id:str)->BaseChatMessageHistory:
    if session_id not in sessionIdStore:
        sessionIdStore[session_id]=ChatMessageHistory()
    return sessionIdStore[session_id] 

/var/folders/t3/g8frt1556l99c9y9bmh3lknh0000gn/T/ipykernel_30477/1099825374.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory


In [21]:
# Wrap in message history

with_message_history=RunnableWithMessageHistory(
    chain,
    getSessionHistory,
    input_messages_key="messages"
)

config={"configurable":{"session_id":"chat5"}}


In [23]:
response=with_message_history.invoke(
    {
        "messages" : messages + [HumanMessage(content="What is my name?")],
        "language" : "English"
    },
    config=config
)

response.content

"I don't know your name, you haven't told me yet! Would you like to share it with me?"

In [ ]:
response=with_message_history.invoke(
    {
        "messages" : messages + [HumanMessage(content="What mathematical problem did I ask?")],
        "language" : "English"
    },
    config=config
)

response.content